
# LightGBM

Цель: обучить LightGBM на пяти готовых фолдах и получить вероятности `satisfaction = 1`. Основная метрика — ROC AUC.

## Порядок работы

1. Загрузить `train_with_folds.parquet` и `test_clean.parquet`.
2. Отделить `satisfaction`, `fold` и `id` от признаков.
3. Преобразовать `Class` в pandas `category` с одинаковым списком категорий в train и test.
4. Создать массивы для OOF и усреднённых test-предсказаний.
5. Обучить пять моделей: текущий `fold` использовать для validation, остальные четыре — для обучения.
6. Использовать early stopping и считать ROC AUC каждого фолда.
7. Записать validation-вероятности в OOF, а test-вероятности усреднить по пяти моделям.

Начальные настройки: `n_estimators=2000`, `learning_rate=0.03`, `num_leaves=31`, `subsample=0.8`, `subsample_freq=1`, `colsample_bytree=0.8`, `reg_lambda=1.0`, `random_state=42`.

При обучении использовать `eval_metric="auc"` и callbacks `early_stopping(100)`, `log_evaluation(100)`. Не создавать новый `train_test_split`, не масштабировать признаки и не заполнять числовые пропуски. Использовать `predict_proba(data)[:, 1]`.

## Результаты

Сохранить:

- `artifacts/oof/lightgbm.npy`;
- `artifacts/test_predictions/lightgbm.npy`;
- `artifacts/metrics/lightgbm.json` с параметрами, fold AUC, OOF AUC и лучшими итерациями.

Проверить длины массивов, диапазон вероятностей от 0 до 1, тип `category` у `Class` и отсутствие `id`, `fold`, `satisfaction` среди признаков.

In [19]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import lightgbm as lgb
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_data_dir = project_root / "data" / "processed"
raw_data_dir = project_root / "data" / "raw"
oof_dir = project_root / "artifacts" / "oof"
test_predictions_dir = project_root / "artifacts" / "test_predictions"
metrics_dir = project_root / "artifacts" / "metrics"

train_path = processed_data_dir / "train_with_folds.parquet"
test_path = processed_data_dir / "test_clean.parquet"
sample_submission_path = raw_data_dir / "sample_submission.csv"

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)
sample_submission_df = pd.read_csv(sample_submission_path)

train_df.shape, test_df.shape, sample_submission_df.shape

((699635, 25), (299844, 23), (299844, 2))

In [3]:
train_df.columns

Index(['id', 'Age', 'Flight Distance', 'Inflight wifi service',
       'Departure/Arrival time convenient', 'Ease of Online booking',
       'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
       'Inflight entertainment', 'On-board service', 'Leg room service',
       'Baggage handling', 'Checkin service', 'Cleanliness',
       'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Gender',
       'Customer Type', 'Type of Travel', 'Class', 'satisfaction',
       'Arrival Delay Missing', 'fold'],
      dtype='str')

In [11]:
target = "satisfaction"

feature_cols = [c for c in train_df.columns
                 if c not in ["satisfaction", "id", "fold"]]

feature_cols, print("Number of features:", len(feature_cols))

Number of features: 22


(['Age',
  'Flight Distance',
  'Inflight wifi service',
  'Departure/Arrival time convenient',
  'Ease of Online booking',
  'Gate location',
  'Food and drink',
  'Online boarding',
  'Seat comfort',
  'Inflight entertainment',
  'On-board service',
  'Leg room service',
  'Baggage handling',
  'Checkin service',
  'Cleanliness',
  'Departure Delay in Minutes',
  'Arrival Delay in Minutes',
  'Gender',
  'Customer Type',
  'Type of Travel',
  'Class',
  'Arrival Delay Missing'],
 None)

In [ ]:
x = train_df[feature_cols]
y = train_df["satisfaction"]

xtest = test_df[feature_cols]

In [15]:
categories = sorted(set(train_df["Class"].dropna().unique()) | set(test_df["Class"].dropna().unique()))

train_df["Class"] = pd.Categorical(train_df["Class"], categories=categories)
test_df["Class"] = pd.Categorical(test_df["Class"], categories=categories)

print(train_df["Class"].dtype)
print(test_df["Class"].dtype)
print(train_df["Class"].cat.categories)

category
category
Index(['Business', 'Eco', 'Eco Plus'], dtype='str')


In [22]:
oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

parameters = {
    "n_estimators": 2000,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": 42
}

model = lgb.LGBMClassifier(**parameters)